# Minerva Transforms: the 1.0 Data Pipeline Pattern

This notebook teaches the pattern every Minerva transform follows. It runs on CPU with synthetic data, so no dataset download is needed.

The rules are short:

1. A **sample is one flat `dict`** (`{"image": ..., "label": ...}`).
2. A transform **takes a dict and returns a dict**.
3. Its first constructor argument is **`keys_to_transform`**. It only touches those keys; every other key passes through unchanged.
4. **Random transforms draw their parameters once per call** and share them across all listed keys, so an image and its label stay aligned. They accept an optional `seed`.
5. Geometric transforms take a per-key interpolation **`mode`**.
6. Arrays use **NumPy's layout**: images are `(H, W)` or `(H, W, C)` (channels last), exactly what the readers return, so you never transpose before transforming.
7. **`MultiViewTransform`** builds independent views (for contrastive learning) as `{view}_{key}`.

Contents:

1. Samples are dicts
2. `keys_to_transform` and pass-through
3. Pipelines
4. Random transforms share their draw
5. Reproducibility with `seed`
6. Per-key interpolation `mode`
7. Multiple views
8. Plugging transforms into a `MinervaDataset`
9. Third-party and tuple-based code
10. Writing your own transform

## Imports

In [1]:
import copy
from collections import OrderedDict

import numpy as np
import torch
from torch.utils.data import DataLoader

from minerva.data.datasets import MinervaDataset, ToTupleDataset
from minerva.data.readers import NumpyArrayReader
from minerva.transforms import (
    CastToTransform,
    ExternalTransform,
    FlipTransform,
    MultiViewTransform,
    NormalizeTransform,
    RandomFlipTransform,
    RandomResizedCropTransform,
    RepeatTransform,
    ResizeTransform,
    SqueezeTransform,
    TransposeTransform,
    _Transform,
    _TransformsPipeline,
)
from minerva.transforms.transform import _RandomTransform

## 1. Samples are dicts

We build a tiny synthetic segmentation sample: a single-channel `image` of shape `(8, 8, 1)` (H, W, C) and an integer `label` mask of shape `(8, 8)` with 3 classes. We also add `meta`, a key no transform will be told about.

In [2]:
rng = np.random.default_rng(0)
sample = {
    "image": rng.random((8, 8, 1)).astype("float32"),
    "label": rng.integers(0, 3, size=(8, 8)).astype("uint8"),
    "meta": "scan-001",
}
{k: getattr(v, "shape", v) for k, v in sample.items()}

{'image': (8, 8, 1), 'label': (8, 8), 'meta': 'scan-001'}

## 2. `keys_to_transform` and pass-through

The first argument of every transform lists the keys it touches. `RepeatTransform(["image"], ...)` repeats the image to 3 channels; `label` and `meta` are not listed, so they come out unchanged.

In [3]:
repeat = RepeatTransform(["image"], axis=-1, n_repetitions=3)  # channels are the last axis
out = repeat(copy.deepcopy(sample))  # transforms update the dict in place; pass a copy

print("image:", out["image"].shape)  # (H, W, C)
print("label unchanged:", np.array_equal(out["label"], sample["label"]))
print("meta unchanged:", out["meta"])

image: (8, 8, 3)
label unchanged: True
meta unchanged: scan-001


> A transform updates and returns the dict it receives. Inside a `MinervaDataset` this is fine, because every `__getitem__` builds a fresh dict. When you call transforms by hand, as here, pass a copy if you want to keep the original.

A key that does not exist raises a plain `KeyError`, so typos fail loudly:

In [4]:
try:
    RepeatTransform(["imagee"], axis=0, n_repetitions=3)({"image": np.zeros((1, 2, 2))})
except KeyError as e:
    print("KeyError:", e)

KeyError: 'imagee'


## 3. Pipelines

`_TransformsPipeline` runs transforms in order. A pipeline is itself a transform, so pipelines can be nested. Each step names its own keys, so one pipeline handles the whole sample.

In [5]:
pipeline = _TransformsPipeline([
    RepeatTransform(["image"], axis=-1, n_repetitions=3),
    NormalizeTransform(["image"], mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25]),
    CastToTransform(["label"], "int64"),
])

out = pipeline(copy.deepcopy(sample))
print(pipeline)
print()
print("image:", out["image"].shape, out["image"].dtype)
print("label:", out["label"].shape, out["label"].dtype)

_TransformsPipeline(transforms=[RepeatTransform(keys_to_transform=['image'], axis=-1, n_repetitions=3), NormalizeTransform(keys_to_transform=['image'], mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25], to_rgb=False), CastToTransform(keys_to_transform=['label'], dtype=int64)])

image: (8, 8, 3) float32
label: (8, 8) int64


## 4. Random transforms share their draw

A random transform draws its parameters **once per call** and applies them to every listed key. Listing `["image", "label"]` therefore flips both the same way, which is what keeps them aligned.

To see this, we make a label that is the image thresholded. If the flip is shared, the relation still holds after the transform, over many calls:

In [6]:
torch.manual_seed(0)
img = rng.random((8, 8, 1)).astype("float32")
sample_aligned = {"image": img, "label": (img[..., 0] > 0.5).astype("uint8")}

# axis 1 is the width of both the (H, W, C) image and the (H, W) label
flip = RandomFlipTransform(["image", "label"], possible_axis=1, prob=0.5)

flipped, aligned = 0, True
for _ in range(200):
    out = flip({k: v.copy() for k, v in sample_aligned.items()})
    aligned &= bool(((out["image"][..., 0] > 0.5).astype("uint8") == out["label"]).all())
    flipped += not np.array_equal(out["image"], sample_aligned["image"])

print(f"flipped in {flipped}/200 calls; image and label aligned every time: {aligned}")

flipped in 109/200 calls; image and label aligned every time: True


Never list several *view* keys on one random transform hoping for different augmentations per view: they would share one draw. Use `MultiViewTransform` for that (section 7).

## 5. Reproducibility with `seed`

Every random transform accepts an optional `seed` (the last argument):

- `seed=None` (default) uses torch's global RNG. `DataLoader` seeds it per worker, and `torch.manual_seed` makes runs reproducible.
- `seed=<int>` gives the transform its own generator, so it is reproducible on its own. Under a `DataLoader`, each worker offsets the seed by its worker id, so workers do not all produce the same augmentation.

In [7]:
def run(seed, n=6):
    t = RandomFlipTransform(["image"], possible_axis=1, prob=0.5, seed=seed)
    base = np.arange(4, dtype="float32").reshape(1, 4)
    return [int(t({"image": base.copy()})["image"][0, 0] == 3) for _ in range(n)]  # 1 = flipped

print("seed=7 :", run(7))
print("seed=7 :", run(7), "(same)")
print("seed=8 :", run(8))

seed=7 : [0, 1, 0, 0, 1, 1]
seed=7 : [0, 1, 0, 0, 1, 1] (same)
seed=8 : [0, 0, 0, 1, 0, 0]


## 6. Per-key interpolation `mode`

Geometric transforms (`ResizeTransform`, `RotationTransform`, `RandomResizedCropTransform`, ...) take a `mode` with **one entry per key**. Images want smooth (`"bilinear"`) interpolation, but labels must stay integer class ids, so they need `"nearest"`.

In [8]:
hwc = {"image": np.repeat(sample["image"], 3, axis=-1), "label": sample["label"]}
resize = ResizeTransform(["image", "label"], 16, 16, mode=["bilinear", "nearest"])
out = resize(copy.deepcopy(hwc))

print("image:", out["image"].shape, "-> values are interpolated floats")
print("label:", out["label"].shape, "-> classes:", np.unique(out["label"]))
print("classes preserved:", set(np.unique(out["label"])) <= {0, 1, 2})

image: (16, 16, 3) -> values are interpolated floats
label: (16, 16) -> classes: [0 1 2]
classes preserved: True


Without `mode`, every key uses bilinear interpolation, which blends class ids into values that belong to no class. Always set `"nearest"` for labels:

In [9]:
bad = ResizeTransform(["label"], 16, 16)({"label": sample["label"].astype("float32")})  # default: bilinear
print("classes with default mode:", np.unique(bad["label"]).round(2)[:6], "...")

classes with default mode: [0.   0.06 0.12 0.19 0.25 0.31] ...


The same applies to random geometric transforms. This one picks a random crop box once and applies it to both keys, with the right interpolation each:

In [10]:
rrc = RandomResizedCropTransform(
    ["image", "label"], size=(16, 16), scale=(0.3, 1.0),
    mode=["bilinear", "nearest"], seed=0,
)
out = rrc(copy.deepcopy(sample))
print(out["image"].shape, out["label"].shape, np.unique(out["label"]))

(16, 16, 1) (16, 16) [0 1 2]


## 7. Multiple views

Contrastive methods (SimCLR, BYOL, SimSiam, ...) need several independent augmentations of one sample. `MultiViewTransform` takes one **sub-pipeline per view**, runs each on its own deep copy of the listed keys, and writes the results as `{view}_{key}`. The source keys are removed.

Every view draws its own random parameters, even when two views use the same pipeline, and different pipelines give asymmetric augmentation.

In [11]:
views = MultiViewTransform(
    ["image"],
    views={
        "view1": _TransformsPipeline([RandomFlipTransform(["image"], possible_axis=1, prob=1.0)]),
        "view2": _TransformsPipeline([RandomFlipTransform(["image"], possible_axis=0, prob=1.0)]),
    },
)
out = views(copy.deepcopy(sample))
print(sorted(out))
print("view1 is flipped on W:", np.array_equal(out["view1_image"], sample["image"][:, ::-1]))
print("view2 is flipped on H:", np.array_equal(out["view2_image"], sample["image"][::-1]))

['label', 'meta', 'view1_image', 'view2_image']
view1 is flipped on W: True
view2 is flipped on H: True


The names `view1_image` / `view2_image` are what Minerva's contrastive models read from the batch, so a `MultiViewTransform` in your dataset is all they need.

## 8. Plugging transforms into a `MinervaDataset`

`MinervaDataset` takes an `OrderedDict` of readers (the key is the dict key of the sample) and **one** transform pipeline. Here the readers wrap in-memory volumes of shape `(8, 8, 10)` and each item is an `(8, 8, 1)` slice; with real data you would use e.g. `NumpyArrayReader(path, ...)` or `TabularReader`.

In [12]:
# Volumes of 10 slices stacked along the LAST axis; each reader item is one (8, 8, 1) slice
images = rng.random((8, 8, 10)).astype("float32")
labels = rng.integers(0, 3, size=(8, 8, 10)).astype("uint8")

dataset = MinervaDataset(
    readers=OrderedDict(
        image=NumpyArrayReader(images, data_shape=(8, 8, 1)),
        label=NumpyArrayReader(labels, data_shape=(8, 8, 1)),
    ),
    transforms=_TransformsPipeline([
        RandomFlipTransform(["image", "label"], possible_axis=1, seed=0),
        RepeatTransform(["image"], axis=-1, n_repetitions=3),
        SqueezeTransform(["label"], axis=-1),
        CastToTransform(["label"], "int64"),
        # models want channels first: do this LAST, once, at the end of the pipeline
        TransposeTransform(["image"], (2, 0, 1)),
    ]),
)
print(dataset)

           📂 MinervaDataset Information           
📌 Dataset Type: MinervaDataset
   └── Reader 'image': NumpyArrayReader(samples=10, shape=(8, 8, 1), dtype=float32)
   └── Reader 'label': NumpyArrayReader(samples=10, shape=(8, 8, 1), dtype=uint8)
   │
   └── Transforms: _TransformsPipeline(transforms=[RandomFlipTransform(seed=0, keys_to_transform=['image', 'label'], possible_axis=1, prob=0.5), RepeatTransform(keys_to_transform=['image'], axis=-1, n_repetitions=3), SqueezeTransform(keys_to_transform=['label'], axis=-1), CastToTransform(keys_to_transform=['label'], dtype=int64), TransposeTransform(keys_to_transform=['image'], axes=(2, 0, 1))])
   └── Total Readers: 2


In [13]:
item = dataset[0]
print({k: (tuple(v.shape), str(v.dtype)) for k, v in item.items()})

{'image': ((3, 8, 8), 'float32'), 'label': ((8, 8), 'int64')}


Batches from a `DataLoader` are dicts of stacked tensors. Models read them by key (`batch["image"]`, `batch["label"]`):

In [14]:
batch = next(iter(DataLoader(dataset, batch_size=4)))
{k: tuple(v.shape) for k, v in batch.items()}

{'image': (4, 3, 8, 8), 'label': (4, 8, 8)}

## 9. Third-party and tuple-based code

**Third-party transforms.** `ExternalTransform(keys, fn)` applies any callable (a torchvision transform, an albumentations call, a lambda) to the listed keys, so it fits in a pipeline like any other transform.

In [15]:
to_range = ExternalTransform(["image"], lambda x: (x - x.min()) / (x.max() - x.min()))
out = to_range({"image": np.array([2.0, 4.0, 6.0]), "label": 1})
print(out)

{'image': array([0. , 0.5, 1. ]), 'label': 1}


**Tuple-based loops.** Code that expects `(data, label)` tuples (a custom loop, another library) can wrap a dict dataset in `ToTupleDataset`. The reverse, `FromTupleDataset`, turns a tuple dataset into a dict dataset.

In [16]:
as_tuples = ToTupleDataset(dataset, data_key="image", target_key="label")
x, y = as_tuples[0]
print(type(x).__name__, tuple(x.shape), type(y).__name__, tuple(y.shape))

ndarray (3, 8, 8) ndarray (8, 8)


## 10. Writing your own transform

Subclass `_Transform` (or `_RandomTransform` if it draws random numbers) and follow the rules:

- the class name ends in `Transform`;
- `keys_to_transform` is the first constructor argument;
- `__call__` takes and returns the dict, and changes only the listed keys.

`_map_keys` applies a function to every listed key. For a random transform, call `self._init_rng(seed)` in the constructor and draw with `self._rand()`, `self._uniform()` or `self._randint()` **once per call**, outside any per-key loop.

In [17]:
class ClipTransform(_Transform):
    """Clip every listed key to [low, high]."""

    def __init__(self, keys_to_transform, low, high):
        self.keys_to_transform = list(keys_to_transform)
        self.low, self.high = low, high

    def __call__(self, data):
        return self._map_keys(data, lambda x: np.clip(x, self.low, self.high))


class RandomScaleTransform(_RandomTransform):
    """Multiply every listed key by ONE random factor in [low, high]."""

    def __init__(self, keys_to_transform, low=0.5, high=1.5, seed=None):
        self._init_rng(seed)
        self.keys_to_transform = list(keys_to_transform)
        self.low, self.high = low, high

    def __call__(self, data):
        factor = self._uniform(self.low, self.high)  # drawn once, shared by all keys
        return self._map_keys(data, lambda x: x * factor)


pipe = _TransformsPipeline([
    RandomScaleTransform(["a", "b"], seed=3),
    ClipTransform(["a"], 0.0, 1.0),
])
out = pipe({"a": np.array([0.8, 0.9]), "b": np.array([0.8, 0.9]), "c": "untouched"})
print(pipe)
print(out)

_TransformsPipeline(transforms=[RandomScaleTransform(seed=3, keys_to_transform=['a', 'b'], low=0.5, high=1.5), ClipTransform(keys_to_transform=['a'], low=0.0, high=1.0)])
{'a': array([0.40341082, 0.45383717]), 'b': array([0.40341082, 0.45383717]), 'c': 'untouched'}


`a` and `b` received the same random factor (they print identical values), only `a` was clipped, and `c` was never touched.

## Summary

| Rule | Why |
|---|---|
| Sample = flat `dict` | one structure for every task; no tuples to unpack |
| `keys_to_transform` first | each step states what it touches; the rest passes through |
| Random draw once per call | image and label stay aligned |
| `seed` on random transforms | reproducible augmentation |
| Per-key `mode` | smooth for images, `"nearest"` for labels |
| `MultiViewTransform` | independent views as `{view}_{key}` for SSL |
| NumPy layout `(H, W, C)` | readers' output goes straight in; transpose once at the end for the model |
| `ExternalTransform` / tuple wrappers | interoperate with existing code |

See `docs/design.md` for the full standard.